# EP13 — NEAT for Pacman Ghost Team (Wittkamp et al.)
**COMPSCI 713 · S1 2025 Q4 · 3 marks**

| Part | Topic | Marks |
|------|-------|-------|
| a | Role of neural network for a ghost + output | 1 |
| b | Why resolve ties in the first fitness rank? | 1 |
| c | Two secondary evaluation measures + their purpose | 1 |

*Reference: Wittkamp et al. — NEAT-based ghost team in Pacman*

---

## Background: NEAT + Pacman

**NEAT (NeuroEvolution of Augmenting Topologies):** A genetic algorithm that evolves both the *weights* AND *topology* of neural networks. Each individual in the population is a neural network.

**The Pacman Setup:**
- 4 computer-controlled ghosts vs the human Pacman
- Maze limits movement to ≤4 directions
- Normally ghosts eat Pacman (take lives). If Pacman eats a power-pill, roles reverse for a few seconds and Pacman can eat ghosts.
- Goal: evolve a ghost team that beats the human player effectively.

---

## Part (a) — Role of the Neural Network [1 mark]

**Role:** Each ghost has its own neural network that acts as its **controller (policy)**. It takes game state information as input and outputs the ghost's next move.

**Inputs** (sensory information the ghost perceives):
- Pacman's position (relative x/y)
- Other ghosts' positions
- Power-pill status (active or not)
- Distance/direction information in the maze

**Output:** A direction choice — one of the valid moves (Up, Down, Left, Right) — that the ghost will take this timestep.

In other words, the neural network is a **behaviour controller**: input = game state, output = action.

---

## Part (b) — Why Resolve Ties in Pacman's Lives? [1 mark]

The first rank metric is **Pacman's number of lives lost**. If many individuals in the population all kill Pacman the same number of times, the first metric produces a tie — and standard selection cannot distinguish them.

**Why this matters:**
If ties aren't broken, ghost strategies that differ in quality but happen to achieve the same kill count are treated as equally good. Natural selection has no signal to distinguish *how* those lives were taken:
- Was it through clever coordinated ambush, or lucky random wandering?
- Did it take 100 steps or 5 steps?

Without tie-breaking, the GA cannot consistently improve *the quality and efficiency* of the hunting strategy — it can only optimise kill count, which quickly plateaus.

Tie-breaking with secondary metrics injects finer-grained signal that guides evolution beyond the primary goal.

---

## Part (c) — Two Secondary Measures [1 mark]

### Measure 1: Time to Catch Pacman (Game Duration / Speed of Capture)
**What it measures:** How quickly (in steps) the ghosts caught Pacman, given the same kill count.

**What the authors are trying to accomplish:** Reward *efficient* hunting. A ghost team that catches Pacman in fewer steps is demonstrating better coordination and strategy than one that stumbles into Pacman by chance after many random moves. This prevents the GA from accepting lazy solutions that get lucky.

### Measure 2: Proximity / Spread of Ghosts to Pacman
**What it measures:** How close the ghosts collectively get to Pacman over time (or how well spread out they are across the maze).

**What the authors are trying to accomplish:** Encourage *teamwork and coordination*. If all 4 ghosts cluster together, they leave most of the maze unguarded — Pacman can easily evade. Rewarding spread encourages the team to corner Pacman from multiple angles, which is the emergent cooperative behaviour the study seeks to evolve.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba

np.random.seed(42)

# =============================================================
# Simulation: NEAT-style evolution for ghost controllers
# Each individual = a small NN that controls one ghost
# Fitness: tiered (lives > speed > coordination)
# =============================================================

def sigmoid(x): return 1 / (1 + np.exp(-x))

class GhostNet:
    """Tiny neural net: 4 inputs (game state) → 4 outputs (move probabilities)."""
    def __init__(self, weights=None):
        # weights shape: 4x4 (input→output, simplified, no hidden layer)
        self.W = weights if weights is not None else np.random.randn(4, 4) * 0.5

    def act(self, state):
        """Return move direction (0=Up,1=Down,2=Left,3=Right)."""
        logits = self.W @ state
        probs  = np.exp(logits) / np.exp(logits).sum()
        return np.argmax(probs)

def simulate_ghost(net, pacman_start=(9, 9), ghost_start=(2, 2), steps=100, maze_size=15):
    """Returns (lives_taken, steps_to_catch)."""
    px, py = pacman_start
    gx, gy = ghost_start
    lives_taken    = 0
    step_to_catch  = steps  # default: never caught

    DIRS = [(0,1),(0,-1),(-1,0),(1,0)]  # Up, Down, Left, Right

    for t in range(steps):
        # State: relative position + power-pill (0 = off)
        state = np.array([px - gx, py - gy,
                          np.sign(px - gx), np.sign(py - gy)])
        move  = net.act(state)
        dx, dy = DIRS[move]

        # Ghost moves
        gx = max(0, min(maze_size, gx + dx))
        gy = max(0, min(maze_size, gy + dy))

        # Pacman takes a random step
        rdx, rdy = DIRS[np.random.randint(4)]
        px = max(0, min(maze_size, px + rdx))
        py = max(0, min(maze_size, py + rdy))

        if abs(gx-px) + abs(gy-py) <= 1:  # caught
            lives_taken  += 1
            step_to_catch = t + 1
            break

    return lives_taken, step_to_catch

# ------- Evolution -------
POP   = 40
GENS  = 30

pop       = [GhostNet() for _ in range(POP)]
best_hist = []
mean_hist = []

for gen in range(GENS):
    fitness = []
    for ind in pop:
        lives, speed = simulate_ghost(ind)
        # Tiered fitness: primary=lives, secondary=1/speed (faster catch = better)
        f = lives * 1000 + (1 / (speed + 1)) * 100
        fitness.append(f)

    fitness = np.array(fitness)
    best_hist.append(fitness.max())
    mean_hist.append(fitness.mean())

    # Selection: top 10 elites
    elite_idx = fitness.argsort()[-10:]
    elites    = [pop[i] for i in elite_idx]

    # Reproduce with mutation
    new_pop = list(elites)
    while len(new_pop) < POP:
        parent = elites[np.random.randint(len(elites))]
        child  = GhostNet(parent.W + np.random.randn(4, 4) * 0.15)
        new_pop.append(child)
    pop = new_pop

print('=== Ghost Team NEAT Evolution ===')
print(f'Start fitness (best): {best_hist[0]:.1f}')
print(f'Final fitness (best): {best_hist[-1]:.1f}')
print(f'Improvement: {best_hist[-1]/max(best_hist[0],1):.1f}x')

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(best_hist, label='Best (tiered fitness)', linewidth=2, color='#e74c3c')
ax.plot(mean_hist, label='Mean', linewidth=1.5, linestyle='--', color='#3498db')
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness (lives×1000 + speed bonus)')
ax.set_title('NEAT Ghost Evolution — Tiered Fitness Function')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Visualisation: Why tie-breaking matters
# Shows two ghost strategies with same kill count but different quality
# =============================================================

np.random.seed(10)

def trace_ghost(weights_hint, steps=80, maze_size=12):
    """Simulate and record path."""
    net = GhostNet(weights_hint)
    px, py = 10, 10
    gx, gy = 1, 1
    DIRS = [(0,1),(0,-1),(-1,0),(1,0)]
    g_path = [(gx, gy)]
    p_path = [(px, py)]
    caught_at = None
    for t in range(steps):
        state = np.array([px-gx, py-gy, np.sign(px-gx), np.sign(py-gy)])
        move  = net.act(state)
        dx, dy = DIRS[move]
        gx = max(0, min(maze_size, gx+dx))
        gy = max(0, min(maze_size, gy+dy))
        px = max(0, min(maze_size, px + DIRS[np.random.randint(4)][0]))
        py = max(0, min(maze_size, py + DIRS[np.random.randint(4)][1]))
        g_path.append((gx,gy))
        p_path.append((px,py))
        if abs(gx-px)+abs(gy-py)<=1 and caught_at is None:
            caught_at = t+1
    return g_path, p_path, caught_at

# Strategy A: direct, efficient hunter (weights push toward Pacman)
W_good = np.array([[1.0, 0.0, 0.8, 0.0],
                    [0.0, 1.0, 0.0, 0.8],
                    [-0.5,-0.5, 0.3, 0.3],
                    [0.2, 0.2, 0.2, 0.2]])

# Strategy B: wanderer that eventually catches (random-ish)
W_lazy = np.random.randn(4,4) * 0.1

g_good, p_good, caught_good = trace_ghost(W_good)
g_lazy, p_lazy, caught_lazy = trace_ghost(W_lazy)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, g_path, p_path, caught, title, colour in [
    (axes[0], g_good, p_good, caught_good, 'Strategy A: Efficient Hunter\n(good coordination)', '#e74c3c'),
    (axes[1], g_lazy, p_lazy, caught_lazy, 'Strategy B: Lucky Wanderer\n(same kill count)', '#95a5a6'),
]:
    gx_arr = [p[0] for p in g_path]
    gy_arr = [p[1] for p in g_path]
    px_arr = [p[0] for p in p_path]
    py_arr = [p[1] for p in p_path]

    ax.plot(gx_arr, gy_arr, color=colour, linewidth=2, label='Ghost path', alpha=0.8)
    ax.plot(px_arr, py_arr, color='#f39c12', linewidth=1.5, linestyle='--',
            label='Pacman path', alpha=0.6)
    ax.scatter(*g_path[0], color=colour, s=100, zorder=5, marker='s', label='Ghost start')
    ax.scatter(*p_path[0], color='#f39c12', s=100, zorder=5, marker='o', label='Pacman start')
    if caught:
        ax.scatter(*g_path[caught], color='black', s=200, zorder=6, marker='X', label=f'Caught at step {caught}')
        ax.set_title(f'{title}\nCaught at step {caught}')
    else:
        ax.set_title(f'{title}\nNever caught')
    ax.set_xlim(-0.5, 13)
    ax.set_ylim(-0.5, 13)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlabel('x'); ax.set_ylabel('y')

fig.suptitle('Same kills, different quality — this is why tie-breaking in fitness is essential',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Efficient hunter caught Pacman at step: {caught_good}')
print(f'Lucky wanderer caught Pacman at step:   {caught_lazy}')
print('Without tie-breaking, both would be considered equally fit!')

## Exam Quick-Reference

**(a) Role of the neural network:**  
Each ghost's neural network is its **behaviour controller** (policy). Input = game state (Pacman position, ghost positions, power-pill status). Output = the next movement direction (one of Up/Down/Left/Right).

**(b) Why tie-breaking matters:**  
Many individuals may achieve the same kill count by chance. Without secondary metrics, the GA cannot distinguish efficient strategies from lucky wandering — evolution stalls because selection has no signal to improve quality within the same kill tier.

**(c) Two secondary measures:**
1. **Time/steps to catch Pacman** — rewards speed of capture, favouring efficient coordinated strategies over lucky ones
2. **Ghost spread / proximity to Pacman** — rewards the team for covering the maze evenly and approaching Pacman from multiple directions, encouraging emergent coordination